Open the catalog, find a dataset, load it with xarray, and plot it.

::: {.callout-note}
This page was run against the **sample catalog**, so the numbers and the data are synthetic. On the cluster, point `CATALOG_PATH` at the production catalog.
:::

## Settings

`CATALOG_PATH` is the intake-esm descriptor. `DATA_ROOT` replaces the catalog's path prefix, for example with an HTTPS prefix or the folder you downloaded files into. Leave it `None` on the cluster. Both can also be set with the `WHOI_CMIP_CATALOG` and `WHOI_CMIP_DATA_ROOT` environment variables.

In [1]:
import os
from pathlib import Path

# Sample catalog shipped with the repository (relative to this notebook).
SAMPLE = Path("../../tests/fixtures")

CATALOG_PATH = os.environ.get("WHOI_CMIP_CATALOG", str(SAMPLE / "sample_catalog.json"))
CATALOG_PREFIX = "/proj/cmip6"  # prefix used in the catalog's path column
DATA_ROOT = os.environ.get(
    "WHOI_CMIP_DATA_ROOT",
    str(SAMPLE / "sample_data") if "WHOI_CMIP_CATALOG" not in os.environ else None,
)

## Open the catalog

In [2]:
import intake

cat = intake.open_esm_datastore(CATALOG_PATH)
if DATA_ROOT:
    cat.df["path"] = cat.df["path"].str.replace(CATALOG_PREFIX, DATA_ROOT, regex=False)
cat

,unique
master_id,160
collection,2
path,1090
activity_id,3
institution_id,5
source_id,5
experiment_id,4
member_id,4
table_id,4
variable_id,12


Each row is one file. The facet columns follow the CMIP6 directory layout.

In [3]:
cat.df[
    [
        "source_id",
        "experiment_id",
        "member_id",
        "table_id",
        "variable_id",
        "version",
        "time_range",
    ]
].head()

,source_id,experiment_id,member_id,table_id,variable_id,version,time_range
0,IPSL-CM6A-LR,historical,r1i1p1f1,Amon,pr,v20200703,185001-189912
1,IPSL-CM6A-LR,historical,r1i1p1f1,Amon,pr,v20200703,190001-194912
2,IPSL-CM6A-LR,historical,r1i1p1f1,Amon,pr,v20200703,195001-199912
3,IPSL-CM6A-LR,historical,r1i1p1f1,Amon,pr,v20200703,200001-201412
4,IPSL-CM6A-LR,historical,r1i1p1f1,Amon,psl,v20231026,185001-189912


What is available? `unique()` lists the values of each facet.

In [4]:
cat.unique()[["source_id", "experiment_id", "table_id", "variable_id"]]

source_id          [IPSL-CM6A-LR, MPI-ESM1-2-HR, CESM2, GFDL-ESM4]
experiment_id                      [historical, piControl, ssp585]
table_id                                        [Amon, Omon, Oday]
variable_id      [pr, psl, tas, so, thetao, tos, air, shum, hgt...
dtype: object

## Search

Narrow to monthly near-surface air temperature from CESM2's historical run.

In [5]:
sub = cat.search(
    source_id="CESM2",
    experiment_id="historical",
    member_id="r1i1p1f1",
    table_id="Amon",
    variable_id="tas",
)
sub.df[["variable_id", "version", "time_range"]]

,variable_id,version,time_range
0,tas,v20190308,185001-189912
1,tas,v20190308,190001-194912
2,tas,v20190308,195001-199912
3,tas,v20190308,200001-201412


## Load with xarray

`to_dataset_dict()` opens each matching dataset, joining its files along time. Keys are the same dataset ids the [browser](../browse.qmd) shows.

In [6]:
dsets = sub.to_dataset_dict(progressbar=False)
key, ds = next(iter(dsets.items()))
print(key)
ds

CMIP6.CMIP.NCAR.CESM2.historical.r1i1p1f1.Amon.tas.gn.v20190308


<xarray.Dataset> Size: 269kB
Dimensions:  (time: 1980, lat: 4, lon: 8)
Coordinates:
  * time     (time) object 16kB 1850-01-16 12:00:00 ... 2014-12-16 12:00:00
  * lat      (lat) float64 32B -67.5 -22.5 22.5 67.5
  * lon      (lon) float64 64B 0.0 45.0 90.0 135.0 180.0 225.0 270.0 315.0
Data variables:
    tas      (time, lat, lon) float32 253kB dask.array<chunksize=(600, 4, 8), meta=np.ndarray>
Attributes: (12/25)
    title:                            SYNTHETIC sample data for WHOI CMIP sit...
    comment:                          Not model output. Generated by make_sam...
    activity_id:                      CMIP
    institution_id:                   NCAR
    source_id:                        CESM2
    experiment_id:                    historical
    ...                               ...
    intake_esm_attrs:table_id:        Amon
    intake_esm_attrs:variable_id:     tas
    intake_esm_attrs:grid_label:      gn
    intake_esm_attrs:version:         v20190308
    intake_esm_attrs:_data_format_:   netcdf
    intake_esm_dataset_key:           CMIP6.CMIP.NCAR.CESM2.historical.r1i1p1...

## Plot

Global mean, weighted by the cosine of latitude, as annual means.

In [7]:
import numpy as np
import plotly.express as px

weights = np.cos(np.deg2rad(ds["lat"]))
gm = ds["tas"].weighted(weights).mean(["lat", "lon"])
annual = gm.groupby("time.year").mean().to_series()

fig = px.line(
    x=annual.index,
    y=annual.values,
    labels={"x": "Year", "y": f"tas ({ds['tas'].attrs['units']})"},
    title=f"Global mean {ds['tas'].attrs['long_name']}: {key}",
)
fig.show()

## Next steps

- [Catalog](../catalog.qmd): columns, versions, using the catalog outside VAST.
- [Access](../access.qmd): cluster, SMB share, or HTTPS.